# Lief Data Science Test
---

## Question 1 — JSON Data Transformation

Transform the received JSON into a flat results list by resolving all ID references:
- `gender_id` → gender label
- `adore_car` → car object (brand + country)
- `car_brand` list → brand names
- `color` list → full RGB objects
- `user_id` → country label

In [ ]:
import json

In [ ]:
# ---- Received JSON ----

received = {
    "students": [
        {"user_id": 1, "first_name": "Dorree", "last_name": "Stanlick", "gender_id": "a2", "adore_car": "c4", "car_brand": ["c1", "c4"], "color": ["IndianRed", "LightCoral"]},
        {"user_id": 2, "first_name": "Gert", "last_name": "Fake", "gender_id": "a1", "adore_car": "c1", "car_brand": ["c1", "c2"], "color": ["DarkSalmon", "LightSalmon"]},
        {"user_id": 3, "first_name": "Rene", "last_name": "Dust", "gender_id": "a2", "adore_car": "c5", "car_brand": ["c3", "c5"], "color": ["Salmon", "LightCoral"]},
        {"user_id": 4, "first_name": "Clo", "last_name": "Cordes", "gender_id": "a1", "adore_car": "c4", "car_brand": ["c2", "c4", "c5"], "color": ["DarkSalmon"]},
        {"user_id": 5, "first_name": "Emlynne", "last_name": "Kettley", "gender_id": "a2", "adore_car": "c1", "car_brand": ["c1"], "color": ["IndianRed", "LightSalmon", "LightCoral", "Salmon"]}
    ],
    "gender": [
        {"gender_id": "a1", "label": "male"},
        {"gender_id": "a2", "label": "female"}
    ],
    "cars": [
        {"car_id": "c1", "car_brand": "Subaru", "car_make": "Japan"},
        {"car_id": "c2", "car_brand": "Dodge", "car_make": "United States"},
        {"car_id": "c3", "car_brand": "Volkswagen", "car_make": "Germany"},
        {"car_id": "c4", "car_brand": "Hyundai", "car_make": "Korea"},
        {"car_id": "c5", "car_brand": "Toyota", "car_make": "Japan"}
    ],
    "rgbCode": [
        {"hex": "#CD5C5C", "label": "IndianRed", "rgb": "205, 92, 92"},
        {"hex": "#F08080", "label": "LightCoral", "rgb": "240, 128, 128"},
        {"hex": "#FA8072", "label": "Salmon", "rgb": "250, 128, 114"},
        {"hex": "#E9967A", "label": "DarkSalmon", "rgb": "233, 150, 122"},
        {"hex": "#FFA07A", "label": "LightSalmon", "rgb": "255, 160, 122"}
    ],
    "countries": [
        {"label": "Czech Republic", "user_ids": [{"userId": "1"}, {"userId": "3"}, {"userId": "5"}]},
        {"label": "Colombia", "user_ids": [{"userId": "2"}, {"userId": "4"}]}
    ]
}

In [ ]:
# ---- Build lookup dicts ----

gender_map = {g["gender_id"]: g["label"] for g in received["gender"]}
car_map    = {c["car_id"]: c for c in received["cars"]}
color_map  = {c["label"]: c for c in received["rgbCode"]}

# country: user_id (str) -> country label
country_map = {}
for country in received["countries"]:
    for uid in country["user_ids"]:
        country_map[int(uid["userId"])] = country["label"]

In [ ]:
# ---- Transform each student ----

results = []
for s in received["students"]:
    adore = car_map[s["adore_car"]]
    results.append({
        "user_id":    s["user_id"],
        "first_name": s["first_name"],
        "last_name":  s["last_name"],
        "gender":     gender_map[s["gender_id"]],
        "adore_car":  {"car_brand": adore["car_brand"], "brand_from": adore["car_make"]},
        "car_brand":  [car_map[cid]["car_brand"] for cid in s["car_brand"]],
        "countries":  country_map[s["user_id"]],
        "colors":     [color_map[c] for c in s["color"]]
    })

print(json.dumps(results, indent=2, ensure_ascii=False))

---
## Question 2 — Cosine Similarity

Write `cosine_similarity(vec1, vec2) -> float` from scratch.

$$\text{cosine\_sim} = \frac{A \cdot B}{\|A\| \times \|B\|}$$

Expected output: `0.9926`

In [ ]:
import math
from typing import List


def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    dot     = sum(a * b for a, b in zip(vec1, vec2))
    mag1    = math.sqrt(sum(a ** 2 for a in vec1))
    mag2    = math.sqrt(sum(b ** 2 for b in vec2))
    return round(dot / (mag1 * mag2), 4)


vec1 = [1, 2, 3]
vec2 = [2, 3, 4]
print(cosine_similarity(vec1, vec2))  # => 0.9926

---
## Question 3 — Sentiment Classification Pipeline

Build a mini ML pipeline:
1. **Preprocess** — lowercase, remove punctuation & stopwords, tokenize
2. **Feature Engineering** — TF-IDF
3. **Model** — Logistic Regression (train 80% / test 20%)
4. **Evaluate** — accuracy + confusion matrix

In [ ]:
import requests
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
# ---- 1. Load dataset ----

url = "https://lief-assets-storage.sgp1.cdn.digitaloceanspaces.com/Test/dataset.txt"
dataset = requests.get(url).json()

texts  = [item["text"]  for item in dataset]
labels = [item["label"] for item in dataset]

print(f"Loaded {len(texts)} samples")

In [ ]:
# ---- 2. Preprocessing ----

STOPWORDS = {"is", "the", "and", "a", "an", "of"}


def preprocess(text: str) -> str:
    text = text.lower()                                          # lowercase
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    tokens = text.split()                                        # tokenize
    tokens = [t for t in tokens if t not in STOPWORDS]           # remove stopwords
    return " ".join(tokens)


texts_clean = [preprocess(t) for t in texts]

# preview
for orig, clean in zip(texts[:3], texts_clean[:3]):
    print(f"{orig}  =>  {clean}")

In [ ]:
# ---- 3. Feature Engineering (TF-IDF) ----

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts_clean)
y = labels

print(f"TF-IDF matrix shape: {X.shape}")

In [ ]:
# ---- 4. Train / Test Split ----

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

In [ ]:
# ---- 5. Model Training ----

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

print("Model trained.")

In [ ]:
# ---- 6. Evaluation ----

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"\nConfusion Matrix:")
print(cm)